# Published Layer Notebook - Ledger Activity
**ILLUSTRATIVE EXAMPLE - fully fictional lakehouse, schema, and table names.**

Recreates only the pattern used in the capstone project: a Fabric notebook (PySpark / Spark SQL) that stages raw exported source data, applies business rules, and materializes published Delta tables in the analytics lakehouse. The notebook is executed each morning by a scheduled Fabric Data Pipeline; the copy list is passed in as a pipeline parameter.

In [ ]:
# Parameters cell - populated by the pipeline (loop over copyItems)
copy_items = [
    {"sourceTable": "histops_db.dbo.DispatchLines", "targetTable": "import_stage.DispatchLineHistory"},
    {"sourceTable": "histops_db.dbo.ItemCatalog",   "targetTable": "import_stage.ItemCatalog"}
]

In [ ]:
%%pyspark
# Stage warehouse-related ledger records from the raw exported tables
activity_df = spark.sql("""
    SELECT
        al.GLCODE,
        fh.DOCREF,
        CAST(fh.POSTDATE AS DATE)  AS PostDate,
        SUM(fd.AMOUNTLOCAL)        AS PostedAmount
    FROM corelanding_lh.fin_ledger.FinEntryHeader fh
    JOIN corelanding_lh.fin_ledger.FinEntryDetail fd
         ON fd.HEADERREF = fh.ROWKEY
    JOIN corelanding_lh.fin_ledger.DimBridge db
         ON fd.BRIDGEREF = db.ROWKEY
    JOIN corelanding_lh.fin_ledger.AccountLookup al
         ON db.ACCOUNTREF = al.ROWKEY
    WHERE fd.CATEGORYCODE IN (7, 8)
    GROUP BY al.GLCODE, fh.DOCREF, CAST(fh.POSTDATE AS DATE)
""")

activity_df.createOrReplaceTempView("activity_staged")
print(f"Staged rows: {activity_df.count():,}")

In [ ]:
%%sql
-- Rebuild the published summary from the staged view
DROP TABLE IF EXISTS insights_lh.pub_reports.LedgerActivitySummary;

CREATE TABLE insights_lh.pub_reports.LedgerActivitySummary
USING DELTA AS
SELECT *, current_timestamp() AS RefreshedAt
FROM activity_staged;

In [ ]:
# Copy historical database tables listed in the pipeline parameter
for item in copy_items:
    src, tgt = item["sourceTable"], item["targetTable"]
    df = spark.read.table(src)          # landed by the pipeline Copy activity
    (df.dropDuplicates()
       .write.mode("overwrite")
       .format("delta")
       .saveAsTable(f"insights_lh.{tgt}"))
    print(f"Loaded {src} -> insights_lh.{tgt}: {df.count():,} rows")

In [ ]:
%%sql
-- Post-load validation: duplicate grain check must return 0
SELECT COUNT(*) AS DuplicateGrainRows FROM (
    SELECT GLCODE, DOCREF, PostDate
    FROM insights_lh.pub_reports.LedgerActivitySummary
    GROUP BY GLCODE, DOCREF, PostDate
    HAVING COUNT(*) > 1
)

**Scheduling:** this notebook is invoked by a Fabric Data Pipeline (loop -> Copy Data -> Notebook) that runs each morning at 05:30 UTC, with e-mail failure notification to the pipeline owner. Downstream Power BI reports pick up the refreshed published tables through the semantic model.